# Steam Games Dataset: EDA & Feature Engineering


## 1. Import Libraries & Plot Configuration

Phần này thiết lập môi trường phân tích và style biểu đồ thống nhất cho toàn notebook.


In [ ]:
import ast
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

# Publication-ready plotting configuration
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "figure.figsize": (9, 5),
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.titleweight": "bold",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
})

sns.set_theme(style="whitegrid", context="notebook", palette="deep")

MAIN_COLOR = "#2A6F97"
ACCENT_COLOR = "#E76F51"
GRAY_COLOR = "#6C757D"
GREEN_COLOR = "#2A9D8F"

def format_number(value):
    # Format large numbers for plot annotations.
    if pd.isna(value):
        return ""
    value = float(value)
    if abs(value) >= 1_000_000:
        return f"{value/1_000_000:.1f}M"
    if abs(value) >= 1_000:
        return f"{value/1_000:.1f}K"
    if value.is_integer():
        return f"{int(value)}"
    return f"{value:.2f}"

def finalize_plot(title=None, xlabel=None, ylabel=None, caption=None):
    # Apply consistent final formatting to a matplotlib plot.
    ax = plt.gca()
    if title:
        ax.set_title(title, pad=14)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    sns.despine()
    plt.tight_layout()
    if caption:
        plt.figtext(
            0.01, -0.02, caption,
            ha="left", va="top",
            fontsize=9, color=GRAY_COLOR
        )

def annotate_horizontal_bars(ax, fmt=format_number):
    # Annotate horizontal bar charts.
    for patch in ax.patches:
        width = patch.get_width()
        y = patch.get_y() + patch.get_height() / 2
        ax.text(width, y, f" {fmt(width)}", va="center", ha="left", fontsize=9, color="#333333")

def safe_display(title, obj):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    display(obj)


## 2. Load Raw and Clean Data

Notebook đọc cả hai phiên bản dữ liệu:

- `df_raw`: dữ liệu ban đầu sau khi thu thập.
- `df_clean`: dữ liệu sau preprocessing, dự kiến dùng cho web/recommendation.

Việc so sánh hai phiên bản giúp kiểm tra xem preprocessing có giải quyết đúng các vấn đề chất lượng dữ liệu hay không.


In [ ]:
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "SteamGames.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "SteamGames_cleaned.csv"

# Kaggle fallback when running from the Kaggle dataset.
kaggle_raw_path = Path("/kaggle/input/datasets/newnguyn/steam-game-clean/SteamGames.csv")
kaggle_clean_path = Path("/kaggle/input/datasets/newnguyn/steam-game-clean/SteamGames_cleaned.csv")

if not RAW_PATH.exists() and kaggle_raw_path.exists():
    RAW_PATH = kaggle_raw_path

if not CLEAN_PATH.exists() and kaggle_clean_path.exists():
    CLEAN_PATH = kaggle_clean_path

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_PATH}")
if not CLEAN_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEAN_PATH}")

df_raw = pd.read_csv(RAW_PATH)
df_clean = pd.read_csv(CLEAN_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW path:", RAW_PATH)
print("CLEAN path:", CLEAN_PATH)
print("RAW shape:", df_raw.shape)
print("CLEAN shape:", df_clean.shape)

safe_display("RAW sample", df_raw.head())
safe_display("CLEAN sample", df_clean.head())


## 3. Dataset Overview

Phần overview giúp trả lời các câu hỏi nền tảng:

- Dataset có bao nhiêu dòng/cột?
- Mỗi cột có kiểu dữ liệu gì?
- Mức độ thiếu dữ liệu ra sao?
- Cột nào có cardinality cao, cần xử lý cẩn thận khi đưa vào mô hình?


In [ ]:
def overview_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        empty_count = s.astype(str).str.strip().eq("").sum() if s.dtype == "object" else 0
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "missing_count": int(s.isna().sum()),
            "missing_rate_%": round(s.isna().mean() * 100, 2),
            "empty_string_count": int(empty_count),
            "unique_count": int(s.nunique(dropna=True)),
            "unique_rate_%": round(s.nunique(dropna=True) / max(len(df), 1) * 100, 2),
        })
    return pd.DataFrame(rows)

raw_overview = overview_table(df_raw)
clean_overview = overview_table(df_clean)

safe_display("RAW data overview", raw_overview)
safe_display("CLEAN data overview", clean_overview)

overview_compare = (
    raw_overview[["column", "missing_count", "empty_string_count", "unique_count"]]
    .merge(
        clean_overview[["column", "missing_count", "empty_string_count", "unique_count"]],
        on="column",
        how="outer",
        suffixes=("_raw", "_clean")
    )
    .fillna(0)
)

safe_display("Raw vs clean overview comparison", overview_compare)


**Insight.**  
Overview không chỉ để kiểm tra lỗi mà còn chỉ ra cột nào phù hợp cho từng mục tiêu. Với hệ thống gợi ý game, `tags`, `Description`, `Developers`, `Publishers` là nhóm feature nội dung; `price`, `ReleaseDate`, review và requirement là nhóm feature dùng để lọc, xếp hạng hoặc giải thích kết quả.


## 4. Data Quality Audit

Data quality được kiểm tra theo ba lớp:

1. Missing values (`NaN`).
2. Chuỗi rỗng nhưng không bị pandas tính là missing.
3. Dòng/cột định danh bị trùng.

Điểm quan trọng là clean data phải an toàn khi upload lên Kaggle và đọc lại bằng `pd.read_csv()`.


### 4.1. Missing Values


In [ ]:
def missing_summary(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    result = pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_rate_%": (df.isna().mean().values * 100).round(2),
    })
    return result.sort_values(["missing_count", "missing_rate_%"], ascending=False)

missing_raw = missing_summary(df_raw, "raw")
missing_clean = missing_summary(df_clean, "clean")

safe_display("Missing values in raw data", missing_raw[missing_raw["missing_count"] > 0])
safe_display("Missing values in clean data", missing_clean[missing_clean["missing_count"] > 0])


In [ ]:
missing_plot = pd.concat([
    missing_raw.assign(stage="Raw"),
    missing_clean.assign(stage="Clean"),
])
missing_plot = missing_plot[missing_plot["missing_count"] > 0].copy()

if len(missing_plot) > 0:
    top_missing_cols = (
        missing_plot.groupby("column")["missing_rate_%"]
        .max()
        .sort_values(ascending=False)
        .head(12)
        .index
    )
    plot_df = missing_plot[missing_plot["column"].isin(top_missing_cols)]

    plt.figure(figsize=(9, 5.5))
    ax = sns.barplot(data=plot_df, x="missing_rate_%", y="column", hue="stage", orient="h")
    finalize_plot(
        title="Missing value rate before and after preprocessing",
        xlabel="Missing rate (%)",
        ylabel="Column",
        caption="Lower missing rate in clean data indicates effective preprocessing."
    )
    plt.legend(title="")
    plt.show()
else:
    print("No missing values found in either dataset.")


**Insight.**  
Missing values trong raw data là tín hiệu cho các bước cleaning cần thiết. Tuy nhiên, mục tiêu không phải luôn là fill mọi thứ: các game thiếu `Name`, thiếu nguồn `Developers/Publishers`, hoặc thiếu metadata cốt lõi có thể nên bị loại bỏ vì không đủ tin cậy để đề xuất.


### 4.2. Empty String Check

Một ô `""` có thể không bị tính là `NaN` trong notebook hiện tại, nhưng khi lưu CSV và đọc lại ở notebook khác, pandas có thể hiểu ô trống thành missing value. Vì vậy cần kiểm tra riêng chuỗi rỗng.


In [ ]:
def empty_string_summary(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    object_cols = df.select_dtypes(include="object").columns
    result = []
    for col in object_cols:
        count = int(df[col].astype(str).str.strip().eq("").sum())
        result.append({
            "dataset": dataset_name,
            "column": col,
            "empty_string_count": count,
            "empty_string_rate_%": round(count / max(len(df), 1) * 100, 2),
        })
    return pd.DataFrame(result).sort_values("empty_string_count", ascending=False)

empty_raw = empty_string_summary(df_raw, "raw")
empty_clean = empty_string_summary(df_clean, "clean")

safe_display("Empty strings in raw data", empty_raw[empty_raw["empty_string_count"] > 0])
safe_display("Empty strings in clean data", empty_clean[empty_clean["empty_string_count"] > 0])


**Insight.**  
Nếu clean data còn chuỗi rỗng ở các cột text quan trọng, pipeline có thể vẫn nhìn “hết null” trong notebook hiện tại nhưng phát sinh `NaN` sau khi reload CSV. Đây là lỗi thường gặp khi chuyển từ preprocessing sang triển khai web/Kaggle.


### 4.3. Duplicate Check


In [ ]:
def duplicate_report(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    rows = [{
        "dataset": dataset_name,
        "key": "full_row",
        "duplicate_count": int(df.duplicated().sum()),
    }]

    for key in ["Appid", "Name", "Rank"]:
        if key in df.columns:
            rows.append({
                "dataset": dataset_name,
                "key": key,
                "duplicate_count": int(df[key].duplicated().sum()),
            })

    return pd.DataFrame(rows)

dup_report = pd.concat([
    duplicate_report(df_raw, "raw"),
    duplicate_report(df_clean, "clean")
], ignore_index=True)

safe_display("Duplicate report", dup_report)

plot_dup = dup_report.copy()
if plot_dup["duplicate_count"].sum() > 0:
    plt.figure(figsize=(8, 4.8))
    ax = sns.barplot(data=plot_dup, x="duplicate_count", y="key", hue="dataset", orient="h")
    annotate_horizontal_bars(ax)
    finalize_plot(
        title="Duplicate count by key",
        xlabel="Duplicate count",
        ylabel="Key",
        caption="Business-key duplicates are more important than exact row duplicates for recommendation systems."
    )
    plt.legend(title="")
    plt.show()
else:
    print("No duplicates detected for checked keys.")


**Insight.**  
Exact duplicate rows không phải vấn đề duy nhất. Với dữ liệu game, `Appid`, `Name` và `Rank` là business keys quan trọng hơn. Nếu `Rank` bị trùng do crawl nhiều đợt, có thể tái tạo lại rank theo thứ tự dòng sau preprocessing.


## 5. Format and Type Consistency

Sau data quality, cần kiểm tra các cột có định dạng dễ sai: giá, ngày phát hành, tag và requirement. Đây là các cột thường gây lỗi khi đưa vào filter/search/recommendation.


### 5.1. Price Distribution


In [ ]:
def clean_price_series(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
        .str.strip()
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("Free To Play", "0", regex=False)
        .str.replace("Free", "0", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan}),
        errors="coerce"
    )

raw_price = clean_price_series(df_raw["price"]) if "price" in df_raw.columns else pd.Series(dtype=float)
clean_price = pd.to_numeric(df_clean["price"], errors="coerce") if "price" in df_clean.columns else pd.Series(dtype=float)

price_check = pd.DataFrame({
    "dataset": ["raw", "clean"],
    "invalid_or_missing_price": [int(raw_price.isna().sum()), int(clean_price.isna().sum())],
    "min_price": [raw_price.min(), clean_price.min()],
    "median_price": [raw_price.median(), clean_price.median()],
    "max_price": [raw_price.max(), clean_price.max()],
    "free_games": [int((raw_price == 0).sum()), int((clean_price == 0).sum())],
})

safe_display("Price consistency check", price_check)

df_clean["price_numeric"] = clean_price.fillna(0)


In [ ]:
if len(clean_price.dropna()) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

    sns.histplot(clean_price.dropna(), bins=50, ax=axes[0], color=MAIN_COLOR)
    axes[0].set_title("Raw scale")
    axes[0].set_xlabel("Price")
    axes[0].set_ylabel("Number of games")

    sns.histplot(np.log1p(clean_price.dropna()), bins=50, ax=axes[1], color=GREEN_COLOR)
    axes[1].set_title("Log scale")
    axes[1].set_xlabel("log1p(price)")
    axes[1].set_ylabel("Number of games")

    fig.suptitle("Price distribution in clean data", fontweight="bold", y=1.03)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print("No valid price values for plotting.")


**Insight.**  
Giá game thường lệch phải: phần lớn game miễn phí hoặc giá thấp, trong khi một số ít game/bundle có giá cao hơn nhiều. Đây là outlier hợp lệ về mặt kinh doanh, không nên xóa máy móc. Khi dùng cho ranking, nên cân nhắc biến đổi log hoặc chia price bucket thay vì dùng giá raw.


### 5.2. Release Date and Time Coverage


In [ ]:
raw_release_dt = pd.to_datetime(df_raw["ReleaseDate"], errors="coerce") if "ReleaseDate" in df_raw.columns else pd.Series(dtype="datetime64[ns]")
clean_release_dt = pd.to_datetime(df_clean["ReleaseDate"], errors="coerce") if "ReleaseDate" in df_clean.columns else pd.Series(dtype="datetime64[ns]")

date_check = pd.DataFrame({
    "dataset": ["raw", "clean"],
    "invalid_or_missing_date": [int(raw_release_dt.isna().sum()), int(clean_release_dt.isna().sum())],
    "min_date": [raw_release_dt.min(), clean_release_dt.min()],
    "max_date": [raw_release_dt.max(), clean_release_dt.max()],
})

safe_display("ReleaseDate consistency check", date_check)

df_clean["ReleaseDate_dt"] = clean_release_dt
df_clean["ReleaseYear"] = clean_release_dt.dt.year


In [ ]:
release_year_counts = df_clean["ReleaseYear"].dropna().astype(int).value_counts().sort_index()

if len(release_year_counts) > 0:
    plt.figure(figsize=(10, 4.8))
    sns.lineplot(
        x=release_year_counts.index,
        y=release_year_counts.values,
        marker="o",
        color=MAIN_COLOR,
        linewidth=2
    )
    finalize_plot(
        title="Number of games by release year",
        xlabel="Release year",
        ylabel="Number of games",
        caption="This trend shows whether the dataset is biased toward newer or older Steam titles."
    )
    plt.show()
else:
    print("No valid release year values for plotting.")


**Insight.**  
Release year cho biết độ phủ thời gian của dataset. Nếu dataset lệch về các game mới, recommender có thể ưu tiên xu hướng hiện tại. Nếu có nhiều game cũ, cần thêm freshness feature để cân bằng giữa game kinh điển và game mới.


### 5.3. Tags as Content Metadata


In [ ]:
def split_tags(value):
    if pd.isna(value):
        return []
    return [tag.strip().lower() for tag in str(value).split(",") if tag.strip()]

if "tags" in df_clean.columns:
    clean_tag_lists = df_clean["tags"].apply(split_tags)
    clean_tag_counts = clean_tag_lists.explode().dropna().value_counts()
    df_clean["num_tags"] = clean_tag_lists.apply(len)

    tag_table = clean_tag_counts.head(30).reset_index()
    tag_table.columns = ["tag", "count"]
    safe_display("Top 30 clean tags", tag_table)
    safe_display("Number of tags per game", df_clean["num_tags"].describe())
else:
    clean_tag_lists = pd.Series([[] for _ in range(len(df_clean))], index=df_clean.index)
    clean_tag_counts = pd.Series(dtype=int)
    df_clean["num_tags"] = 0
    print("Column 'tags' not found in clean data.")


**Insight.**  
`tags` là tín hiệu nội dung mạnh nhất cho content-based recommendation. Tuy nhiên, số tag quá ít làm game khó được mô tả chính xác; số tag quá nhiều có thể tạo nhiễu nếu tag không được chuẩn hóa. Vì vậy, tag quality quan trọng hơn số lượng raw tag.


## 6. Categorical Exploration

Mục tiêu của phần này là hiểu cấu trúc các cột phân loại: mức độ đa dạng, nhóm phổ biến, và khả năng dùng trong search/filter/recommendation.


### 6.1. Cardinality and Dominant Values


In [ ]:
cat_cols = df_clean.select_dtypes(include="object").columns

cat_summary = pd.DataFrame({
    "column": cat_cols,
    "unique_count": [df_clean[col].nunique(dropna=True) for col in cat_cols],
    "top_value": [
        df_clean[col].mode(dropna=True).iloc[0] if len(df_clean[col].mode(dropna=True)) > 0 else None
        for col in cat_cols
    ],
    "top_value_count": [
        df_clean[col].value_counts(dropna=True).iloc[0] if len(df_clean[col].value_counts(dropna=True)) > 0 else 0
        for col in cat_cols
    ],
})

cat_summary["top_value_rate_%"] = (cat_summary["top_value_count"] / max(len(df_clean), 1) * 100).round(2)
safe_display("Categorical cardinality summary", cat_summary.sort_values("unique_count", ascending=False))


**Insight.**  
Cột cardinality cao như `Name`, `Description`, `Developers`, `Publishers` không nên one-hot trực tiếp toàn bộ trong mô hình đơn giản. Với recommender, các cột này phù hợp hơn khi đưa vào text representation hoặc embedding.


### 6.2. Top Tags


In [ ]:
if len(clean_tag_counts) > 0:
    top_tags = clean_tag_counts.head(20).reset_index()
    top_tags.columns = ["Tag", "Count"]

    plt.figure(figsize=(9, 7))
    ax = sns.barplot(data=top_tags, x="Count", y="Tag", color=MAIN_COLOR)
    annotate_horizontal_bars(ax)
    finalize_plot(
        title="Top 20 most common tags",
        xlabel="Number of games",
        ylabel="Tag",
        caption="Tags define the main content vocabulary used by filters and content-based recommendation."
    )
    plt.show()

    safe_display("Top 20 tags table", top_tags)
else:
    print("No tags available for plotting.")


**Insight.**  
Các tag phổ biến cho thấy ngôn ngữ nội dung chính của dataset. Nếu một vài tag quá thống trị, recommender có thể tạo kết quả quá rộng; khi đó cần kết hợp thêm description, developer/publisher hoặc weighting theo tag rarity.


### 6.3. Tag Richness per Game


In [ ]:
if "num_tags" in df_clean.columns:
    max_tags = int(df_clean["num_tags"].max()) if len(df_clean) else 0
    bins = range(0, max_tags + 2) if max_tags < 80 else 40

    plt.figure(figsize=(8.5, 4.8))
    sns.histplot(df_clean["num_tags"], bins=bins, color=GREEN_COLOR)
    finalize_plot(
        title="Distribution of tag count per game",
        xlabel="Number of tags",
        ylabel="Number of games",
        caption="Tag richness measures how much explicit metadata each game has."
    )
    plt.show()

    safe_display("Tag richness statistics", df_clean["num_tags"].describe())
else:
    print("num_tags not available.")


**Insight.**  
Tag richness là một feature hữu ích: game có nhiều tag thường dễ được mô tả bằng nội dung hơn. Tuy nhiên, nhiều tag không đồng nghĩa với chất lượng cao nếu tag bị trùng nghĩa hoặc quá chung chung.


### 6.4. Top Developers and Publishers


In [ ]:
for col in ["Developers", "Publishers"]:
    if col in df_clean.columns:
        top_values = df_clean[col].value_counts().head(15).reset_index()
        top_values.columns = [col, "Count"]

        plt.figure(figsize=(9, 6))
        ax = sns.barplot(data=top_values, x="Count", y=col, color=MAIN_COLOR)
        annotate_horizontal_bars(ax)
        finalize_plot(
            title=f"Top 15 {col}",
            xlabel="Number of games",
            ylabel=col,
            caption=f"{col} can add useful context for recommendation, but high-cardinality values should not be one-hot encoded naively."
        )
        plt.show()

        safe_display(f"Top 15 {col}", top_values)


**Insight.**  
Developer/publisher là tín hiệu phong cách và hệ sinh thái sản phẩm. Tuy nhiên, đây là cột high-cardinality, nên phù hợp hơn với text feature hoặc target/frequency encoding thay vì one-hot trực tiếp toàn bộ.


### 6.5. Free vs Paid Games


In [ ]:
if "price_numeric" in df_clean.columns:
    df_clean["price_type"] = np.where(df_clean["price_numeric"] == 0, "Free", "Paid")
    price_type_counts = df_clean["price_type"].value_counts().reset_index()
    price_type_counts.columns = ["price_type", "count"]
    price_type_counts["share_%"] = (price_type_counts["count"] / len(df_clean) * 100).round(2)

    plt.figure(figsize=(6, 4.5))
    ax = sns.barplot(data=price_type_counts, x="price_type", y="count", color=MAIN_COLOR)
    for container in ax.containers:
        ax.bar_label(container, fmt="%d", padding=3)
    finalize_plot(
        title="Free vs paid games",
        xlabel="Price type",
        ylabel="Number of games",
        caption="Free/paid status is a simple but useful filter for web users."
    )
    plt.show()

    safe_display("Free vs paid table", price_type_counts)


**Insight.**  
Free/paid là feature dễ hiểu với người dùng và có thể dùng trực tiếp trong giao diện filter. Với ranking, không nên mặc định xem game miễn phí là tốt hơn; price nên được kết hợp với quality/review signals.


## 7. Numerical Exploration

Phần này phân tích giá, review và các quan hệ định lượng. Với dữ liệu game, review thường có phân phối long-tail nên cần dùng log scale để nhìn cấu trúc rõ hơn.


### 7.1. Review Distribution


In [ ]:
review_cols = [col for col in ["PositiveReview", "NegativeReview"] if col in df_clean.columns]

for col in review_cols:
    values = pd.to_numeric(df_clean[col], errors="coerce").fillna(0)
    df_clean[col] = values

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

    sns.histplot(values, bins=50, ax=axes[0], color=MAIN_COLOR)
    axes[0].set_title(f"{col} - raw scale")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Number of games")

    sns.histplot(np.log1p(values), bins=50, ax=axes[1], color=GREEN_COLOR)
    axes[1].set_title(f"{col} - log scale")
    axes[1].set_xlabel(f"log1p({col})")
    axes[1].set_ylabel("Number of games")

    fig.suptitle(f"Distribution of {col}", fontweight="bold", y=1.03)
    sns.despine()
    plt.tight_layout()
    plt.show()

    safe_display(f"{col} statistics", values.describe())


**Insight.**  
Review count có long-tail mạnh: đa số game có rất ít review, một số ít game blockbuster chiếm lượng review rất lớn. Đây không phải lỗi dữ liệu, mà là đặc điểm tự nhiên của nền tảng game. Nếu ranking chỉ dựa trên review count, hệ thống sẽ bị popularity bias.


### 7.2. Review Ratio and Total Reviews


In [ ]:
if {"PositiveReview", "NegativeReview"}.issubset(df_clean.columns):
    df_clean["PositiveReview"] = pd.to_numeric(df_clean["PositiveReview"], errors="coerce").fillna(0)
    df_clean["NegativeReview"] = pd.to_numeric(df_clean["NegativeReview"], errors="coerce").fillna(0)
    df_clean["TotalReviews"] = df_clean["PositiveReview"] + df_clean["NegativeReview"]
    df_clean["ReviewRatio"] = np.where(
        df_clean["TotalReviews"] > 0,
        df_clean["PositiveReview"] / df_clean["TotalReviews"],
        np.nan,
    )

    safe_display(
        "Review metrics statistics",
        df_clean[["PositiveReview", "NegativeReview", "TotalReviews", "ReviewRatio"]].describe()
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

    sns.histplot(df_clean["ReviewRatio"].dropna(), bins=50, ax=axes[0], color=MAIN_COLOR)
    axes[0].set_title("Review ratio distribution")
    axes[0].set_xlabel("PositiveReview / TotalReviews")
    axes[0].set_ylabel("Number of games")

    sns.scatterplot(
        data=df_clean,
        x=np.log1p(df_clean["TotalReviews"]),
        y="ReviewRatio",
        alpha=0.35,
        ax=axes[1],
        color=ACCENT_COLOR,
        edgecolor=None
    )
    axes[1].set_title("Review ratio vs review volume")
    axes[1].set_xlabel("log1p(TotalReviews)")
    axes[1].set_ylabel("Review ratio")

    fig.suptitle("Quality signal versus confidence signal", fontweight="bold", y=1.03)
    sns.despine()
    plt.tight_layout()
    plt.show()


**Insight.**  
Review ratio đo chất lượng cảm nhận, còn total reviews đo độ tin cậy của tín hiệu đó.

$$
ReviewRatio = \frac{PositiveReview}{PositiveReview + NegativeReview}
$$

Một game có 100% positive nhưng chỉ vài review không nên được xếp ngang với game có hàng chục nghìn review. Đây là lý do cần Bayesian smoothing ở phần feature engineering.


### 7.3. Price vs Positive Reviews


In [ ]:
if {"price_numeric", "PositiveReview"}.issubset(df_clean.columns):
    plt.figure(figsize=(8, 5.2))
    sns.scatterplot(
        data=df_clean,
        x="price_numeric",
        y=np.log1p(df_clean["PositiveReview"]),
        alpha=0.35,
        color=MAIN_COLOR,
        edgecolor=None
    )
    finalize_plot(
        title="Price vs positive reviews",
        xlabel="Price",
        ylabel="log1p(PositiveReview)",
        caption="The relationship should be interpreted carefully because review volume is dominated by popularity and exposure."
    )
    plt.show()


**Insight.**  
Giá không phản ánh trực tiếp chất lượng. Một game giá cao có thể ít review vì kén người chơi, còn game miễn phí có thể nhiều review do dễ tiếp cận. Vì vậy, price nên được dùng như filter hoặc feature phụ, không nên là tín hiệu ranking độc lập.


### 7.4. Correlation Heatmap


In [ ]:
numeric_cols = [
    col for col in df_clean.select_dtypes(include=[np.number]).columns
    if df_clean[col].nunique(dropna=True) > 1
]

if len(numeric_cols) >= 2:
    corr = df_clean[numeric_cols].corr()

    plt.figure(figsize=(11, 8))
    sns.heatmap(
        corr,
        cmap="vlag",
        center=0,
        linewidths=0.4,
        linecolor="white",
        cbar_kws={"shrink": 0.8},
        annot=False
    )
    finalize_plot(
        title="Correlation heatmap of numeric features",
        xlabel="",
        ylabel="",
        caption="Correlation is descriptive, not causal; it helps detect redundant or related numeric features."
    )
    plt.show()

    safe_display("Correlation matrix", corr)
else:
    print("Not enough numeric columns for correlation heatmap.")


**Insight.**  
Correlation heatmap giúp phát hiện feature trùng thông tin, ví dụ `PositiveReview`, `NegativeReview` và `TotalReviews` thường liên quan mạnh. Với mô hình tuyến tính, cần tránh đưa quá nhiều feature gần tương đương mà không kiểm soát.


## 8. Long-tail and Popularity Bias

Phần này trả lời câu hỏi phản biện quan trọng: nếu chỉ đề xuất game phổ biến, hệ thống có đang bỏ qua phần lớn catalog không?


In [ ]:
if "TotalReviews" in df_clean.columns and df_clean["TotalReviews"].sum() > 0:
    sorted_reviews = df_clean["TotalReviews"].sort_values(ascending=False).reset_index(drop=True)
    cumulative_share = sorted_reviews.cumsum() / sorted_reviews.sum()
    game_share = np.arange(1, len(sorted_reviews) + 1) / len(sorted_reviews)

    plt.figure(figsize=(8.5, 5.2))
    sns.lineplot(x=game_share, y=cumulative_share, color=MAIN_COLOR, linewidth=2.2)
    plt.axhline(0.8, linestyle="--", linewidth=1.2, color=GRAY_COLOR)
    plt.axvline(0.2, linestyle="--", linewidth=1.2, color=GRAY_COLOR)
    finalize_plot(
        title="Cumulative review share by games",
        xlabel="Share of games",
        ylabel="Share of total reviews",
        caption="A steep curve indicates long-tail concentration: a small share of games receives most reviews."
    )
    plt.show()

    top_20pct_review_share = cumulative_share.iloc[max(int(len(cumulative_share) * 0.2) - 1, 0)]
    print(f"Top 20% games account for approximately {top_20pct_review_share:.2%} of all reviews.")
else:
    print("TotalReviews is not available or all values are zero.")


**Insight.**  
Long-tail là đặc điểm có giá trị phản biện: hệ thống recommendation không nên chỉ tối ưu theo popularity vì sẽ làm giảm khả năng khám phá game ít nổi hơn. Do đó, feature ranking cần cân bằng giữa chất lượng, độ tin cậy, độ mới và nội dung phù hợp.


## 9. Advanced Feature Engineering

Phần này chuyển các insight EDA thành feature có thể dùng cho web/recommendation.

Các nhóm feature được tạo:

1. **Content features**: tag richness, content text, multi-hot tags.
2. **Quality features**: review ratio, Bayesian rating.
3. **Confidence/popularity features**: total reviews, popularity score.
4. **Temporal features**: game age, freshness decay.
5. **Business features**: price bucket, affordability/value score.
6. **Composite ranking features**: discovery score cân bằng giữa quality, confidence, freshness và price.


### 9.1. Content Features: Tag Multi-hot and Weighted Text


In [ ]:
if "tags" in df_clean.columns:
    tag_encoded = df_clean["tags"].astype(str).str.get_dummies(sep=", ")
    tag_encoded = tag_encoded.add_prefix("Tag_")
    df_clean["TagRichness"] = df_clean["num_tags"]
else:
    tag_encoded = pd.DataFrame(index=df_clean.index)
    df_clean["TagRichness"] = 0

def safe_text_col(df, col):
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)

# Weighted text: tags are repeated to give them stronger influence in TF-IDF similarity.
df_clean["ContentText"] = (
    (safe_text_col(df_clean, "tags") + " ") * 3
    + safe_text_col(df_clean, "Name") + " "
    + safe_text_col(df_clean, "Description") + " "
    + safe_text_col(df_clean, "Developers") + " "
    + safe_text_col(df_clean, "Publishers")
).str.replace(r"\s+", " ", regex=True).str.strip()

print("Tag encoded shape:", tag_encoded.shape)
safe_display("Tag multi-hot sample", tag_encoded.head())
safe_display("ContentText sample", df_clean[["Name", "ContentText"]].head())


**Feature rationale.**  
Tag multi-hot phù hợp cho filter và mô hình tuyến tính đơn giản. `ContentText` phù hợp cho TF-IDF/cosine similarity vì kết hợp tag, mô tả, developer và publisher trong một representation thống nhất. Việc lặp lại `tags` giúp metadata đã chuẩn hóa có trọng số cao hơn text mô tả dài.


### 9.2. Bayesian Rating: Quality with Confidence


In [ ]:
if {"ReviewRatio", "TotalReviews"}.issubset(df_clean.columns):
    C = df_clean["ReviewRatio"].mean(skipna=True)
    m = df_clean["TotalReviews"].quantile(0.75)

    def calculate_bayesian_rating(row, m, C):
        v = row["TotalReviews"]
        R = row["ReviewRatio"]

        if pd.isna(R) or v <= 0:
            return C

        return (v / (v + m) * R) + (m / (v + m) * C)

    df_clean["ReviewConfidence"] = 1 - np.exp(-df_clean["TotalReviews"] / max(m, 1))
    df_clean["BayesianRating"] = df_clean.apply(
        lambda row: calculate_bayesian_rating(row, m, C),
        axis=1,
    )

    top_bayesian = (
        df_clean[["Name", "PositiveReview", "NegativeReview", "TotalReviews", "ReviewRatio", "ReviewConfidence", "BayesianRating"]]
        .sort_values("BayesianRating", ascending=False)
        .head(15)
    )

    safe_display("Top games by BayesianRating", top_bayesian)

    plt.figure(figsize=(8.5, 4.8))
    sns.histplot(df_clean["BayesianRating"].dropna(), bins=40, color=MAIN_COLOR)
    finalize_plot(
        title="Bayesian rating distribution",
        xlabel="BayesianRating",
        ylabel="Number of games",
        caption="Bayesian smoothing reduces overconfidence for games with very few reviews."
    )
    plt.show()
else:
    print("ReviewRatio and TotalReviews are required for BayesianRating.")


**Feature rationale.**  
Bayesian rating giải quyết điểm yếu của review ratio raw: một game ít review nhưng 100% positive sẽ bị kéo về trung bình chung, trong khi game có nhiều review ổn định sẽ giữ được điểm cao. Đây là feature phù hợp hơn cho ranking.


### 9.3. Freshness Score with Time Decay


In [ ]:
if "ReleaseDate_dt" in df_clean.columns:
    current_year = pd.Timestamp.today().year
    df_clean["GameAge"] = current_year - df_clean["ReleaseYear"]
    df_clean["GameAge"] = df_clean["GameAge"].clip(lower=0)

    # Exponential decay: newer games get higher freshness, but old games do not become zero.
    HALF_LIFE_YEARS = 5
    df_clean["FreshnessScore"] = np.exp(-np.log(2) * df_clean["GameAge"] / HALF_LIFE_YEARS)

    safe_display(
        "Freshness feature sample",
        df_clean[["Name", "ReleaseDate", "ReleaseYear", "GameAge", "FreshnessScore"]].head()
    )

    plt.figure(figsize=(8.5, 4.8))
    sns.histplot(df_clean["GameAge"].dropna(), bins=40, color=GREEN_COLOR)
    finalize_plot(
        title="Game age distribution",
        xlabel="Game age in years",
        ylabel="Number of games",
        caption="Freshness score can help balance classic games and recent releases."
    )
    plt.show()
else:
    print("ReleaseDate is required for freshness features.")


**Feature rationale.**  
Freshness không nên là rule cứng kiểu “game mới tốt hơn game cũ”. Exponential decay cho phép game mới có lợi thế nhẹ, nhưng game cũ vẫn có thể được đề xuất nếu content/quality phù hợp.


### 9.4. Popularity, Price Bucket, and Value Features


In [ ]:
def minmax_robust(series, lower_q=0.01, upper_q=0.99):
    s = pd.to_numeric(series, errors="coerce")
    lo = s.quantile(lower_q)
    hi = s.quantile(upper_q)
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    clipped = s.clip(lo, hi)
    return (clipped - lo) / (hi - lo)

if "TotalReviews" in df_clean.columns:
    df_clean["PopularityScore"] = minmax_robust(np.log1p(df_clean["TotalReviews"]))
else:
    df_clean["PopularityScore"] = 0

if "price_numeric" in df_clean.columns:
    df_clean["PriceBucket"] = pd.cut(
        df_clean["price_numeric"],
        bins=[-0.01, 0, 5, 15, 30, np.inf],
        labels=["Free", "Budget", "Low-mid", "Mid", "Premium"]
    )

    # Affordable score: free and cheaper games get higher affordability.
    df_clean["AffordabilityScore"] = 1 - minmax_robust(np.log1p(df_clean["price_numeric"]))
else:
    df_clean["PriceBucket"] = "Unknown"
    df_clean["AffordabilityScore"] = 0

bucket_counts = df_clean["PriceBucket"].value_counts(dropna=False).reset_index()
bucket_counts.columns = ["PriceBucket", "Count"]
safe_display("Price bucket distribution", bucket_counts)

feature_cols = [col for col in ["PopularityScore", "AffordabilityScore"] if col in df_clean.columns]
safe_display("Popularity and affordability features", df_clean[["Name"] + feature_cols].head())


**Feature rationale.**  
Popularity score dùng review volume nhưng đã log-transform để giảm ảnh hưởng của blockbuster. Affordability score không nói game rẻ là tốt hơn, mà là một tín hiệu phụ phục vụ web ranking/filter theo ngữ cảnh người dùng.


### 9.5. Composite Discovery Score

`DiscoveryScore` là feature tổng hợp để minh họa cách cân bằng nhiều tín hiệu:

- Quality: `BayesianRating`
- Confidence/popularity: `PopularityScore`
- Freshness: `FreshnessScore`
- Affordability: `AffordabilityScore`

Đây không phải “điểm chân lý”, mà là baseline ranking có thể giải thích được.


In [ ]:
required_score_cols = ["BayesianRating", "PopularityScore", "FreshnessScore", "AffordabilityScore"]

for col in required_score_cols:
    if col not in df_clean.columns:
        df_clean[col] = 0

# Fill remaining missing values with neutral score.
for col in required_score_cols:
    median_value = df_clean[col].median(skipna=True)
    if pd.isna(median_value):
        median_value = 0
    df_clean[col] = df_clean[col].fillna(median_value)

df_clean["DiscoveryScore"] = (
    0.50 * df_clean["BayesianRating"]
    + 0.25 * df_clean["PopularityScore"]
    + 0.15 * df_clean["FreshnessScore"]
    + 0.10 * df_clean["AffordabilityScore"]
)

top_discovery = (
    df_clean[[
        "Name", "price_numeric", "BayesianRating",
        "PopularityScore", "FreshnessScore", "AffordabilityScore", "DiscoveryScore"
    ]]
    .sort_values("DiscoveryScore", ascending=False)
    .head(15)
)

safe_display("Top games by DiscoveryScore", top_discovery)

plt.figure(figsize=(8.5, 4.8))
sns.histplot(df_clean["DiscoveryScore"].dropna(), bins=40, color=MAIN_COLOR)
finalize_plot(
    title="Discovery score distribution",
    xlabel="DiscoveryScore",
    ylabel="Number of games",
    caption="Composite scores should be interpreted as ranking heuristics, not as ground-truth quality."
)
plt.show()


### 9.6. Save Useful Engineered Features

Ch? l?u c?c feature nh? v? c? ?ch cho ranking/filter/web. C?c representation n?ng nh? `ContentText`, `tag_list`, `num_tags`, `GameAge` v?n ch? d?ng trong EDA.


In [ ]:
selected_engineered_features = [
    "price_numeric",
    "price_type",
    "TotalReviews",
    "ReviewRatio",
    "ReviewConfidence",
    "BayesianRating",
    "PopularityScore",
    "FreshnessScore",
    "AffordabilityScore",
    "PriceBucket",
    "DiscoveryScore",
]

base_clean_df = pd.read_csv(CLEAN_PATH)
features_to_save = [col for col in selected_engineered_features if col in df_clean.columns]

# Remove previous versions first so rerunning this cell stays idempotent.
base_clean_df = base_clean_df.drop(columns=[col for col in selected_engineered_features if col in base_clean_df.columns], errors="ignore")
updated_clean_df = base_clean_df.join(df_clean[features_to_save])

feature_null_counts = updated_clean_df[features_to_save].isna().sum()
if int(feature_null_counts.sum()) > 0:
    raise ValueError(f"Selected engineered features still contain nulls:\n{feature_null_counts[feature_null_counts > 0]}")

updated_clean_df.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")
print("Saved selected engineered features to:", CLEAN_PATH)
print("Saved features:", features_to_save)
print("Updated shape:", updated_clean_df.shape)

safe_display("Saved engineered feature sample", updated_clean_df[["Name"] + features_to_save].head())


**Feature rationale.**  
Composite score giúp web có baseline ranking dễ giải thích. Điểm này phản biện lại cách xếp hạng chỉ theo popularity: một game được đề xuất tốt hơn khi vừa có chất lượng ổn định, vừa đủ độ tin cậy, vẫn còn tính thời sự và phù hợp về giá.


### 9.7. Optional: TF-IDF Representation for Content-based Recommendation

Cell này tạo representation cho recommender. Nếu chạy ở môi trường nhẹ, có thể dùng trực tiếp `ContentText` + TF-IDF + cosine similarity trong backend.


In [ ]:
try:

    max_features = min(20000, max(1000, len(df_clean) * 10))
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2
    )

    tfidf_matrix = vectorizer.fit_transform(df_clean["ContentText"].fillna(""))
    print("TF-IDF matrix shape:", tfidf_matrix.shape)

    # Compact semantic representation for potential downstream models.
    n_components = min(50, max(2, tfidf_matrix.shape[1] - 1), max(2, tfidf_matrix.shape[0] - 1))
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    content_svd = svd.fit_transform(tfidf_matrix)

    print("SVD content representation shape:", content_svd.shape)
    print("Explained variance ratio:", round(svd.explained_variance_ratio_.sum(), 4))

except Exception as e:
    print("TF-IDF/SVD feature generation skipped:", e)


**Feature rationale.**  
TF-IDF phù hợp cho version đầu của content-based recommender vì đơn giản, minh bạch và dễ triển khai. SVD có thể nén ma trận sparse thành vector compact nếu cần dùng cho mô hình khác hoặc visualization.


## 10. Embedding and Vector Quality Evaluation

Sau khi tạo các feature nội dung, bước tiếp theo là kiểm tra xem vector representation có thật sự hữu ích cho recommendation hay không. Phần này không chỉ nhìn vào hình học của vector, mà còn kiểm tra xem các game gần nhau trong không gian vector có giống nhau theo các tiêu chí domain hay không.

Trọng tâm đánh giá gồm:

1. **Vector geometry**: TF-IDF/SVD có tạo được không gian biểu diễn ổn định hay không.
2. **Nearest-neighbor quality**: các game gần nhau có trùng tag, developer hoặc publisher nhiều hơn random baseline không.
3. **Qualitative sanity check**: một vài game mẫu có được gợi ý sang các game hợp lý về nội dung hay không.

Cách đọc phần này: nếu nearest-neighbor tốt hơn random baseline rõ rệt, embedding có tín hiệu thực tế để dùng làm baseline recommender cho web.


### 10.1. Embedding Construction Recap

Embedding baseline được xây từ `ContentText`, trong đó `tags` được lặp lại nhiều lần hơn so với description/developer/publisher để nhấn mạnh tín hiệu thể loại và gameplay đã chuẩn hóa. Sau đó:

- TF-IDF tạo vector sparse từ văn bản.
- SVD nén vector sparse thành không gian dense 50 chiều để dễ visualize và truy vấn nearest-neighbor nhanh hơn.

Output cho thấy dataset có khoảng **29.750 game**, TF-IDF dùng **20.000 chiều**, sparse rate khoảng **99,31%**, và SVD 50 chiều giữ khoảng **13,91% explained variance**. Mức explained variance này không quá cao, nhưng vẫn chấp nhận được với text/tag data vì mục tiêu chính là local similarity, không phải tái tạo toàn bộ ma trận TF-IDF.


In [ ]:

# Rebuild representation if the previous TF-IDF/SVD cell was skipped or variables are unavailable.
if "ContentText" not in df_clean.columns:
    tag_text = df_clean["tags"].fillna("").astype(str) if "tags" in df_clean.columns else ""
    description_text = df_clean["Description"].fillna("").astype(str) if "Description" in df_clean.columns else ""
    developer_text = df_clean["Developers"].fillna("").astype(str) if "Developers" in df_clean.columns else ""
    publisher_text = df_clean["Publishers"].fillna("").astype(str) if "Publishers" in df_clean.columns else ""
    df_clean["ContentText"] = (
        (tag_text + " ") * 4
        + description_text + " "
        + developer_text + " "
        + publisher_text
    )

need_rebuild = "tfidf_matrix" not in globals() or "content_svd" not in globals()

if need_rebuild:
    max_features = min(20000, max(1000, len(df_clean) * 10))
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2
    )
    tfidf_matrix = vectorizer.fit_transform(df_clean["ContentText"].fillna(""))

    n_components = min(80, max(2, tfidf_matrix.shape[1] - 1), max(2, tfidf_matrix.shape[0] - 1))
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    content_svd = svd.fit_transform(tfidf_matrix)

embedding_summary = pd.DataFrame({
    "metric": [
        "Number of games",
        "TF-IDF dimensions",
        "TF-IDF non-zero values",
        "TF-IDF sparsity rate",
        "SVD dimensions",
        "SVD explained variance"
    ],
    "value": [
        len(df_clean),
        tfidf_matrix.shape[1],
        tfidf_matrix.nnz,
        round(1 - tfidf_matrix.nnz / max(tfidf_matrix.shape[0] * tfidf_matrix.shape[1], 1), 4),
        content_svd.shape[1],
        round(float(svd.explained_variance_ratio_.sum()), 4) if "svd" in globals() else np.nan
    ]
})

safe_display("Embedding representation summary", embedding_summary)


**Insight.**  
TF-IDF rất sparse là hiện tượng bình thường với dữ liệu text/tag. SVD chỉ giữ khoảng 13,91% variance, nên không nên xem biểu diễn 50 chiều này là embedding semantic hoàn chỉnh. Tuy nhiên, kết quả nearest-neighbor ở phần sau mới là bằng chứng quan trọng hơn: nếu các hàng xóm gần nhất vẫn chia sẻ tag/developer/publisher nhiều hơn random, vector vẫn có giá trị thực tế cho recommender baseline.


### 10.2. Vector Space Geometry

Biểu đồ 2D bên dưới dùng hai thành phần SVD đầu tiên để kiểm tra cấu trúc tổng quát của không gian vector. Không nên diễn giải khoảng cách 2D quá tuyệt đối, vì phần lớn thông tin vẫn nằm ở nhiều chiều hơn. Mục tiêu của biểu đồ là phát hiện nhanh các cụm lớn, điểm cô lập, hoặc embedding bị collapse.

In [ ]:
plot_sample_size = min(2500, len(df_clean))
plot_df = df_clean.sample(plot_sample_size, random_state=42).copy() if len(df_clean) > plot_sample_size else df_clean.copy()
plot_indices = plot_df.index.to_numpy()

embedding_2d = content_svd[plot_indices, :2]
plot_df["EmbeddingDim1"] = embedding_2d[:, 0]
plot_df["EmbeddingDim2"] = embedding_2d[:, 1]

if "TagRichness" not in plot_df.columns:
    plot_df["TagRichness"] = plot_df["tags"].fillna("").astype(str).apply(
        lambda x: len([tag.strip() for tag in x.split(",") if tag.strip()])
    )

plt.figure(figsize=(8.5, 6))
scatter = sns.scatterplot(
    data=plot_df,
    x="EmbeddingDim1",
    y="EmbeddingDim2",
    hue="TagRichness",
    size="TagRichness",
    sizes=(12, 55),
    alpha=0.65,
    linewidth=0,
    palette="viridis"
)
finalize_plot(
    title="2D projection of content embeddings",
    xlabel="SVD component 1",
    ylabel="SVD component 2",
    caption="Color and size represent tag richness; the plot is a diagnostic projection, not the full embedding space."
)
plt.legend(title="Tag richness", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()


**Insight.**  
Biểu đồ 2D chỉ là lát cắt trực quan của không gian nhiều chiều, nên không dùng để kết luận tuyệt đối. Nếu thấy một số vùng dày đặc và một số điểm tách xa, điều đó gợi ý rằng metadata/tag đang tạo cấu trúc nhất định trong không gian vector. Các điểm cô lập nên được hiểu là game có nội dung rất đặc thù hoặc metadata nghèo, cần kiểm tra thêm khi recommender trả kết quả lạ.


### 10.3. Nearest-neighbor Quality Metrics

Một embedding tốt cho recommendation không chỉ cần cosine similarity cao. Hàng xóm gần nhất cũng phải giống game gốc theo các tín hiệu có thể giải thích được:

- **Cosine similarity**: độ gần trong không gian vector.
- **Tag Jaccard similarity**: mức độ overlap giữa hai tập tag.
- **Same developer / publisher rate**: tỷ lệ hàng xóm có cùng nguồn sản xuất/phát hành.

Trong output, nearest-neighbor vượt random baseline rất rõ: cosine trung bình khoảng **0,8766 vs 0,1891**, tag Jaccard khoảng **0,4880 vs 0,0773**, same developer rate khoảng **0,1313 vs 0,0001**, và same publisher rate khoảng **0,1530 vs 0,0017**. Điều này cho thấy embedding không chỉ gom các dòng gần nhau về mặt toán học, mà còn giữ được tín hiệu domain quan trọng.


In [ ]:
def parse_tag_set(value):
    return set(
        tag.strip().lower()
        for tag in str(value).split(",")
        if tag.strip()
    )

def jaccard_similarity(a, b):
    if not a and not b:
        return 0.0
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)

# Use a manageable sample for evaluation to keep notebook runtime stable.
eval_size = min(700, len(df_clean))
eval_indices = np.random.default_rng(42).choice(len(df_clean), size=eval_size, replace=False)

# Nearest neighbors in dense SVD space.
k_neighbors = min(11, len(df_clean))
nn_model = NearestNeighbors(n_neighbors=k_neighbors, metric="cosine", algorithm="brute")
nn_model.fit(content_svd)

distances, neighbor_indices = nn_model.kneighbors(content_svd[eval_indices])

all_tag_sets = df_clean["tags"].fillna("").astype(str).apply(parse_tag_set).tolist()
all_developers = df_clean["Developers"].fillna("").astype(str).str.lower().str.strip().tolist() if "Developers" in df_clean.columns else [""] * len(df_clean)
all_publishers = df_clean["Publishers"].fillna("").astype(str).str.lower().str.strip().tolist() if "Publishers" in df_clean.columns else [""] * len(df_clean)

neighbor_rows = []
for row_pos, source_idx in enumerate(eval_indices):
    # Skip first neighbor because it is usually the item itself.
    for rank, (nbr_idx, dist) in enumerate(zip(neighbor_indices[row_pos][1:], distances[row_pos][1:]), start=1):
        neighbor_rows.append({
            "source_index": int(source_idx),
            "neighbor_index": int(nbr_idx),
            "neighbor_rank": rank,
            "cosine_similarity": float(1 - dist),
            "tag_jaccard": jaccard_similarity(all_tag_sets[source_idx], all_tag_sets[nbr_idx]),
            "same_developer": int(all_developers[source_idx] != "" and all_developers[source_idx] == all_developers[nbr_idx]),
            "same_publisher": int(all_publishers[source_idx] != "" and all_publishers[source_idx] == all_publishers[nbr_idx]),
        })

neighbor_quality = pd.DataFrame(neighbor_rows)

# Random baseline with the same number of pairs.
rng = np.random.default_rng(42)
random_a = rng.integers(0, len(df_clean), size=len(neighbor_quality))
random_b = rng.integers(0, len(df_clean), size=len(neighbor_quality))
random_cosine = cosine_similarity(content_svd[random_a], content_svd[random_b]).diagonal()

random_rows = []
for a, b, sim in zip(random_a, random_b, random_cosine):
    random_rows.append({
        "cosine_similarity": float(sim),
        "tag_jaccard": jaccard_similarity(all_tag_sets[a], all_tag_sets[b]),
        "same_developer": int(all_developers[a] != "" and all_developers[a] == all_developers[b]),
        "same_publisher": int(all_publishers[a] != "" and all_publishers[a] == all_publishers[b]),
    })

random_quality = pd.DataFrame(random_rows)

quality_summary = pd.DataFrame({
    "metric": [
        "Mean cosine similarity",
        "Mean tag Jaccard similarity",
        "Same developer rate",
        "Same publisher rate"
    ],
    "Nearest neighbors": [
        neighbor_quality["cosine_similarity"].mean(),
        neighbor_quality["tag_jaccard"].mean(),
        neighbor_quality["same_developer"].mean(),
        neighbor_quality["same_publisher"].mean(),
    ],
    "Random baseline": [
        random_quality["cosine_similarity"].mean(),
        random_quality["tag_jaccard"].mean(),
        random_quality["same_developer"].mean(),
        random_quality["same_publisher"].mean(),
    ]
})

quality_summary["Lift"] = quality_summary["Nearest neighbors"] / quality_summary["Random baseline"].replace(0, np.nan)
quality_summary = quality_summary.round(4)

safe_display("Embedding quality: nearest neighbors vs random baseline", quality_summary)


In [ ]:
quality_plot = quality_summary.melt(
    id_vars="metric",
    value_vars=["Nearest neighbors", "Random baseline"],
    var_name="pair_type",
    value_name="score"
)

plt.figure(figsize=(9, 5.4))
ax = sns.barplot(
    data=quality_plot,
    x="score",
    y="metric",
    hue="pair_type",
    orient="h"
)
annotate_horizontal_bars(ax)
finalize_plot(
    title="Embedding quality compared with random baseline",
    xlabel="Average score",
    ylabel="Metric",
    caption="A useful recommender embedding should produce higher local semantic similarity than random pairs."
)
plt.legend(title="Pair type")
plt.tight_layout()
plt.show()


In [ ]:
similarity_distribution = pd.DataFrame({
    "cosine_similarity": pd.concat([
        neighbor_quality["cosine_similarity"].sample(min(3000, len(neighbor_quality)), random_state=42),
        random_quality["cosine_similarity"].sample(min(3000, len(random_quality)), random_state=42)
    ], ignore_index=True),
    "pair_type": (
        ["Nearest neighbors"] * min(3000, len(neighbor_quality))
        + ["Random baseline"] * min(3000, len(random_quality))
    )
})

plt.figure(figsize=(8.5, 5))
sns.kdeplot(
    data=similarity_distribution,
    x="cosine_similarity",
    hue="pair_type",
    fill=True,
    alpha=0.25,
    common_norm=False
)
finalize_plot(
    title="Cosine similarity distribution",
    xlabel="Cosine similarity",
    ylabel="Density",
    caption="Separation between neighbor and random distributions suggests the vector space contains usable local structure."
)
plt.tight_layout()
plt.show()


**Insight.**  
Nearest-neighbor có lift lớn so với random baseline ở cả cosine, tag overlap và production metadata. Đây là tín hiệu tốt: baseline `ContentText` + TF-IDF/SVD đã đủ mạnh để làm version đầu của content-based recommender. Tuy vậy, same developer/publisher vẫn thấp tuyệt đối vì Steam catalog rất phân mảnh và developer/publisher có cardinality cao; do đó hai tín hiệu này nên được xem là bonus, không phải điều kiện bắt buộc.


### 10.4. Qualitative Sanity Check: Example Recommendations

Metric tổng hợp vẫn chưa đủ. Một recommender cần được kiểm tra bằng ví dụ cụ thể: với một game đầu vào, các game gần nhất có cùng thể loại, cơ chế chơi, hoặc publisher/developer liên quan hay không.

In [ ]:
def get_neighbors_for_game(game_idx, top_n=5):
    distances, indices = nn_model.kneighbors(content_svd[[game_idx]], n_neighbors=min(top_n + 1, len(df_clean)))
    rows = []
    source_tags = all_tag_sets[game_idx]

    for rank, (nbr_idx, dist) in enumerate(zip(indices[0], distances[0])):
        if nbr_idx == game_idx:
            continue
        rows.append({
            "source_game": df_clean.loc[game_idx, "Name"] if "Name" in df_clean.columns else game_idx,
            "neighbor_rank": rank,
            "recommended_game": df_clean.loc[nbr_idx, "Name"] if "Name" in df_clean.columns else nbr_idx,
            "cosine_similarity": round(float(1 - dist), 4),
            "tag_jaccard": round(jaccard_similarity(source_tags, all_tag_sets[nbr_idx]), 4),
            "neighbor_tags": df_clean.loc[nbr_idx, "tags"] if "tags" in df_clean.columns else "",
        })

        if len(rows) >= top_n:
            break

    return pd.DataFrame(rows)

if "DiscoveryScore" in df_clean.columns:
    example_indices = df_clean.sort_values("DiscoveryScore", ascending=False).head(3).index.tolist()
else:
    example_indices = df_clean.sample(min(3, len(df_clean)), random_state=42).index.tolist()

example_neighbors = pd.concat(
    [get_neighbors_for_game(idx, top_n=5) for idx in example_indices],
    ignore_index=True
)

safe_display("Qualitative nearest-neighbor sanity check", example_neighbors)


**Interpretation guide.**  
Bảng sanity check cho thấy các recommendation thường giữ được chủ đề rộng của game gốc, đặc biệt ở những game có tag rõ. Ví dụ các game kinh dị hoặc co-op có xu hướng được kéo về nhóm cùng không khí/cơ chế chơi. Tuy nhiên vẫn có một số kết quả cosine cao nhưng tag Jaccard trung bình, cho thấy baseline text embedding nên được kết hợp thêm rule/filter theo tag chính khi đưa lên web.


### 10.5. Embedding Readiness Assessment

Với output hiện tại, embedding baseline từ `ContentText` đạt điều kiện để dùng làm recommender ban đầu:

- Cosine similarity của nearest-neighbor cao hơn random baseline khoảng **4,64 lần**.
- Tag Jaccard cao hơn random baseline khoảng **6,31 lần**.
- Same developer và same publisher cao hơn random rõ rệt, dù tỷ lệ tuyệt đối không lớn.
- Kiểm tra thủ công cho thấy nhiều neighbor vẫn cùng nhóm nội dung.

Kết luận thực dụng: baseline này đủ tốt cho web prototype. Khi triển khai, nên dùng thêm filter/rerank theo `tags`, giá, OS requirement hoặc Bayesian rating để giảm các recommendation “gần về text nhưng chưa đúng gu”.


## 11. Structured JSON Embedding and Vector Comparison

Phần trước dùng `ContentText`, tức là một chuỗi văn bản phẳng. Cách đó đơn giản và hiệu quả, nhưng chưa phản ánh cách một hệ thống production có thể lưu mỗi game như một document có cấu trúc.

Ở phần này, mỗi game được chuyển thành một **JSON document** gồm identity, content, production, commercial, requirements và engineered signals. Sau đó notebook so sánh hai chiến lược embedding:

1. **Plain text embedding** từ `ContentText`.
2. **Structured JSON embedding** từ `GameJSONText`.

Mục tiêu không phải giả định JSON chắc chắn tốt hơn, mà là kiểm chứng: việc thêm schema/field name có cải thiện chất lượng neighbor hay chỉ làm vector nhiễu hơn.


### 11.1. Build One JSON Document per Game

Mỗi game được serialize thành một JSON document có cấu trúc. Document này giữ các nhóm thông tin quan trọng:

- `identity`: tên game, loại game, năm phát hành.
- `content`: description và tags.
- `production`: developer và publisher.
- `commercial`: price và price bucket.
- `requirements`: OS, RAM, CPU.
- `engineered_signals`: tag richness, review ratio, Bayesian rating, popularity/freshness/discovery score.

Output ví dụ cho thấy JSON document chứa nhiều tín hiệu hơn plain text. Tuy nhiên, nhiều tín hiệu hơn không tự động đồng nghĩa với embedding tốt hơn; nếu JSON quá dài hoặc description lấn át tag, vector có thể bị phân tán.


In [ ]:

# Ensure required engineered columns exist even if this section is run independently.
if "tag_list" not in df_clean.columns:
    df_clean["tag_list"] = df_clean["tags"].fillna("no tags").astype(str).apply(
        lambda x: [tag.strip() for tag in x.split(",") if tag.strip()]
    )

if "ReleaseYear" not in df_clean.columns:
    df_clean["ReleaseDate_dt"] = pd.to_datetime(df_clean["ReleaseDate"], errors="coerce")
    df_clean["ReleaseYear"] = df_clean["ReleaseDate_dt"].dt.year

if "TotalReviews" not in df_clean.columns:
    df_clean["TotalReviews"] = df_clean["PositiveReview"] + df_clean["NegativeReview"]

if "ReviewRatio" not in df_clean.columns:
    df_clean["ReviewRatio"] = np.where(
        df_clean["TotalReviews"] > 0,
        df_clean["PositiveReview"] / df_clean["TotalReviews"],
        np.nan
    )

if "BayesianRating" not in df_clean.columns:
    C = df_clean["ReviewRatio"].mean(skipna=True)
    m = df_clean["TotalReviews"].quantile(0.75)
    df_clean["BayesianRating"] = np.where(
        df_clean["TotalReviews"] > 0,
        (df_clean["TotalReviews"] / (df_clean["TotalReviews"] + m) * df_clean["ReviewRatio"].fillna(C))
        + (m / (df_clean["TotalReviews"] + m) * C),
        C
    )

if "TagRichness" not in df_clean.columns:
    df_clean["TagRichness"] = df_clean["tag_list"].apply(len)

if "GameAge" not in df_clean.columns:
    current_year = pd.Timestamp.today().year
    df_clean["GameAge"] = (current_year - df_clean["ReleaseYear"]).clip(lower=0)

if "FreshnessScore" not in df_clean.columns:
    df_clean["FreshnessScore"] = np.exp(-df_clean["GameAge"].fillna(df_clean["GameAge"].median()) / 8)

if "PopularityScore" not in df_clean.columns:
    df_clean["PopularityScore"] = np.log1p(df_clean["TotalReviews"])

if "DiscoveryScore" not in df_clean.columns:
    # Balanced signal: quality, enough engagement, and recency.
    quality = df_clean["BayesianRating"].fillna(df_clean["BayesianRating"].median())
    popularity = df_clean["PopularityScore"].fillna(0)
    freshness = df_clean["FreshnessScore"].fillna(df_clean["FreshnessScore"].median())
    df_clean["DiscoveryScore"] = quality * np.log1p(popularity) * (0.7 + 0.3 * freshness)

if "PriceBucket" not in df_clean.columns:
    df_clean["PriceBucket"] = pd.cut(
        df_clean["price"].fillna(0),
        bins=[-0.01, 0, 5, 15, 30, np.inf],
        labels=["free", "budget", "mid", "premium", "high"],
    ).astype(str)


def safe_value(value, default="unknown"):
    if pd.isna(value):
        return default
    value = str(value).strip()
    return value if value else default


def build_game_json(row):
    game_doc = {
        "identity": {
            "name": safe_value(row.get("Name")),
            "type": safe_value(row.get("Type")),
            "release_year": None if pd.isna(row.get("ReleaseYear")) else int(row.get("ReleaseYear")),
        },
        "content": {
            "description": safe_value(row.get("Description"), "no description"),
            "tags": row.get("tag_list", []),
        },
        "production": {
            "developers": safe_value(row.get("Developers"), "unknown developer"),
            "publishers": safe_value(row.get("Publishers"), "unknown publisher"),
        },
        "commercial": {
            "price": float(row.get("price", 0)),
            "price_bucket": safe_value(row.get("PriceBucket"), "unknown"),
        },
        "requirements": {
            "os": safe_value(row.get("OsRequirement"), "unknown"),
            "memory_mb": float(row.get("MemoryRequirement", 0)),
            "cpu_ghz": float(row.get("CPU_GHz", 0)),
            "cpu_tier": int(row.get("CPU_tier", 0)),
        },
        "engineered_signals": {
            "tag_richness": int(row.get("TagRichness", 0)),
            "review_ratio": None if pd.isna(row.get("ReviewRatio")) else round(float(row.get("ReviewRatio")), 4),
            "bayesian_rating": round(float(row.get("BayesianRating", 0)), 4),
            "total_reviews_log": round(float(row.get("PopularityScore", 0)), 4),
            "freshness_score": round(float(row.get("FreshnessScore", 0)), 4),
            "discovery_score": round(float(row.get("DiscoveryScore", 0)), 4),
        },
    }
    return game_doc


def compact_json_text(game_doc):
    return json.dumps(game_doc, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


df_clean["GameJSON"] = df_clean.apply(build_game_json, axis=1)
df_clean["GameJSONText"] = df_clean["GameJSON"].apply(compact_json_text)

json_preview = df_clean[["Name", "GameJSONText"]].head(3)
display(json_preview)
print("Example JSON document:")
print(json.dumps(df_clean["GameJSON"].iloc[0], ensure_ascii=False, indent=2))


### 11.2. Generate Embeddings for Plain Text and JSON Documents

Notebook tạo embedding cho hai representation:

1. **Plain text embedding**: embed `ContentText`, phù hợp làm baseline vì tập trung vào tags, description, developer và publisher.
2. **Structured JSON embedding**: embed `GameJSONText`, giữ field name và các engineered signals.

Nếu môi trường có `sentence-transformers`, notebook dùng semantic embedding thật. Nếu không, notebook fallback sang TF-IDF + SVD để vẫn chạy được. Trong cả hai trường hợp, phần đánh giá phía sau mới quyết định representation nào phù hợp hơn cho bài toán hiện tại.


In [ ]:

# Limit is only for notebook speed. Set to None to embed the full dataset.
EMBED_SAMPLE_LIMIT = min(3000, len(df_clean))
embedding_df = df_clean.sample(EMBED_SAMPLE_LIMIT, random_state=42).reset_index(drop=False).rename(columns={"index": "original_index"})

plain_text_docs = embedding_df["ContentText"].fillna("").astype(str).tolist() if "ContentText" in embedding_df.columns else (
    embedding_df["Name"].fillna("").astype(str) + " " +
    embedding_df["Description"].fillna("").astype(str) + " " +
    embedding_df["tags"].fillna("").astype(str) + " " +
    embedding_df["Developers"].fillna("").astype(str) + " " +
    embedding_df["Publishers"].fillna("").astype(str)
).tolist()

json_docs = embedding_df["GameJSONText"].fillna("").astype(str).tolist()


def build_sentence_transformer_embeddings(docs, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        docs,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return np.asarray(embeddings, dtype=np.float32), model_name


def build_tfidf_svd_embeddings(docs, n_components=128):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=40000,
        ngram_range=(1, 2),
        min_df=2,
    )
    tfidf = vectorizer.fit_transform(docs)
    max_components = max(2, min(n_components, tfidf.shape[1] - 1, tfidf.shape[0] - 1))
    svd = TruncatedSVD(n_components=max_components, random_state=42)
    dense = svd.fit_transform(tfidf)
    dense = normalize(dense)
    return dense, f"TF-IDF + SVD fallback ({max_components} dimensions)", svd.explained_variance_ratio_.sum()

try:
    plain_embeddings, plain_backend = build_sentence_transformer_embeddings(plain_text_docs)
    json_embeddings, json_backend = build_sentence_transformer_embeddings(json_docs)
    embedding_backend = f"SentenceTransformer: {json_backend}"
    json_explained_variance = np.nan
    plain_explained_variance = np.nan
except Exception as e:
    print("SentenceTransformer embedding was not available. Falling back to TF-IDF + SVD.")
    print("Reason:", repr(e))
    plain_embeddings, plain_backend, plain_explained_variance = build_tfidf_svd_embeddings(plain_text_docs)
    json_embeddings, json_backend, json_explained_variance = build_tfidf_svd_embeddings(json_docs)
    embedding_backend = "TF-IDF + SVD fallback"

embedding_summary = pd.DataFrame([
    {
        "representation": "Plain ContentText",
        "backend": plain_backend,
        "n_games": plain_embeddings.shape[0],
        "embedding_dim": plain_embeddings.shape[1],
        "explained_variance_if_svd": plain_explained_variance,
    },
    {
        "representation": "Structured GameJSONText",
        "backend": json_backend,
        "n_games": json_embeddings.shape[0],
        "embedding_dim": json_embeddings.shape[1],
        "explained_variance_if_svd": json_explained_variance,
    },
])

display(embedding_summary)


### 11.3. Compare Vector Spaces by Domain-Aware Neighbor Quality

Chất lượng embedding được so sánh bằng các metric domain-aware:

- **Mean cosine**: mức độ gần trong không gian vector.
- **Mean tag Jaccard**: mức độ trùng tag giữa game gốc và neighbor.
- **Same developer/publisher rate**: khả năng giữ tín hiệu production.
- **Median price/Bayesian/discovery gap**: độ gần về profile sản phẩm.

Output hiện tại cho thấy **plain text embedding đang tốt hơn structured JSON embedding** trên các tín hiệu nội dung chính:

- Plain text có tag Jaccard khoảng **0,2534**, trong khi JSON khoảng **0,1736**.
- Plain text có same developer/publisher rate cao hơn JSON.
- JSON chỉ nhỉnh hơn rất nhẹ ở `median_bayesian_gap` (**0,0228 vs 0,0236**), nhưng không đủ để bù việc giảm tag overlap.

Điều này là insight quan trọng: với dữ liệu hiện tại, JSON document chưa tự động tạo representation tốt hơn. Plain text được thiết kế có trọng số tag rõ ràng nên giữ tín hiệu nội dung tốt hơn.


In [ ]:
def tag_jaccard(tags_a, tags_b):
    set_a = set(tags_a) if isinstance(tags_a, list) else set()
    set_b = set(tags_b) if isinstance(tags_b, list) else set()
    if not set_a and not set_b:
        return 0
    return len(set_a & set_b) / len(set_a | set_b)


def neighbor_indices(embeddings, top_n=10):
    n_neighbors = min(top_n + 1, len(embeddings))
    model = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine")
    model.fit(embeddings)
    distances, indices = model.kneighbors(embeddings)
    return distances[:, 1:], indices[:, 1:]


def evaluate_neighbor_space(embeddings, frame, label, top_n=10, sample_size=500):
    distances, indices = neighbor_indices(embeddings, top_n=top_n)
    rng = np.random.default_rng(42)
    sample_size = min(sample_size, len(frame))
    sample_indices = rng.choice(len(frame), size=sample_size, replace=False)

    rows = []
    for source_idx in sample_indices:
        source = frame.iloc[source_idx]
        source_tags = source.get("tag_list", [])

        for rank, neighbor_idx in enumerate(indices[source_idx], start=1):
            neighbor = frame.iloc[neighbor_idx]
            rows.append({
                "representation": label,
                "source_index": source_idx,
                "neighbor_index": int(neighbor_idx),
                "rank": rank,
                "cosine_similarity": 1 - distances[source_idx][rank - 1],
                "tag_jaccard": tag_jaccard(source_tags, neighbor.get("tag_list", [])),
                "same_developer": safe_value(source.get("Developers")) == safe_value(neighbor.get("Developers")),
                "same_publisher": safe_value(source.get("Publishers")) == safe_value(neighbor.get("Publishers")),
                "price_gap": abs(float(source.get("price", 0)) - float(neighbor.get("price", 0))),
                "bayesian_gap": abs(float(source.get("BayesianRating", 0)) - float(neighbor.get("BayesianRating", 0))),
                "discovery_gap": abs(float(source.get("DiscoveryScore", 0)) - float(neighbor.get("DiscoveryScore", 0))),
            })

    return pd.DataFrame(rows), indices


plain_quality, plain_neighbor_idx = evaluate_neighbor_space(
    plain_embeddings,
    embedding_df,
    label="Plain text embedding",
    top_n=10,
)

json_quality, json_neighbor_idx = evaluate_neighbor_space(
    json_embeddings,
    embedding_df,
    label="Structured JSON embedding",
    top_n=10,
)

embedding_quality = pd.concat([plain_quality, json_quality], ignore_index=True)

quality_comparison = (
    embedding_quality
    .groupby("representation")
    .agg(
        mean_cosine=("cosine_similarity", "mean"),
        mean_tag_jaccard=("tag_jaccard", "mean"),
        same_developer_rate=("same_developer", "mean"),
        same_publisher_rate=("same_publisher", "mean"),
        median_price_gap=("price_gap", "median"),
        median_bayesian_gap=("bayesian_gap", "median"),
        median_discovery_gap=("discovery_gap", "median"),
    )
    .reset_index()
)

display(quality_comparison)

In [ ]:
quality_for_plot = quality_comparison.melt(
    id_vars="representation",
    value_vars=[
        "mean_tag_jaccard",
        "same_developer_rate",
        "same_publisher_rate",
        "mean_cosine",
    ],
    var_name="metric",
    value_name="score",
)

metric_labels = {
    "mean_tag_jaccard": "Tag Jaccard",
    "same_developer_rate": "Same developer",
    "same_publisher_rate": "Same publisher",
    "mean_cosine": "Cosine similarity",
}
quality_for_plot["metric"] = quality_for_plot["metric"].map(metric_labels)

plt.figure(figsize=(10, 5.8), dpi=180)
ax = sns.barplot(
    data=quality_for_plot,
    x="score",
    y="metric",
    hue="representation",
    orient="h",
)
ax.set_title("Structured JSON embedding vs plain text embedding", pad=12, weight="bold")
ax.set_xlabel("Average score")
ax.set_ylabel("")
ax.legend(title="Representation", frameon=False, loc="upper right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
gap_for_plot = quality_comparison.melt(
    id_vars="representation",
    value_vars=[
        "median_price_gap",
        "median_bayesian_gap",
        "median_discovery_gap",
    ],
    var_name="metric",
    value_name="gap",
)

gap_labels = {
    "median_price_gap": "Price gap",
    "median_bayesian_gap": "Bayesian rating gap",
    "median_discovery_gap": "Discovery score gap",
}
gap_for_plot["metric"] = gap_for_plot["metric"].map(gap_labels)

plt.figure(figsize=(10, 5.8), dpi=180)
ax = sns.barplot(
    data=gap_for_plot,
    x="gap",
    y="metric",
    hue="representation",
    orient="h",
)
ax.set_title("Median difference between source games and nearest neighbors", pad=12, weight="bold")
ax.set_xlabel("Median absolute difference")
ax.set_ylabel("")
ax.legend(title="Representation", frameon=False, loc="lower right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=9)
plt.tight_layout()
plt.show()

### 11.4. How Different Are the Two Embedding Strategies?

Nếu JSON embedding và plain-text embedding trả về cùng neighbor, JSON chỉ đang tái tạo baseline. Nếu hai cách khác nhau nhưng JSON giữ domain metric tốt hơn, JSON có thể có giá trị thực tế.

Output cho thấy overlap giữa hai không gian khá thấp: trung bình **overlap@5 ≈ 0,216** và **overlap@10 ≈ 0,232**. Nghĩa là JSON embedding thật sự tạo ra neighborhood khác plain text, nhưng ở phần 11.3, neighborhood đó chưa tốt hơn về tag/developer/publisher.

Diễn giải phản biện: JSON representation đang thay đổi kết quả, nhưng thay đổi đó chưa chắc là cải thiện. Cần tinh chỉnh schema hoặc trọng số field trước khi dùng JSON embedding thay baseline.


In [ ]:
def neighbor_overlap_at_k(indices_a, indices_b, k=10):
    overlaps = []
    for row_a, row_b in zip(indices_a, indices_b):
        set_a = set(row_a[:k])
        set_b = set(row_b[:k])
        overlaps.append(len(set_a & set_b) / k)
    return np.asarray(overlaps)

neighbor_overlap = pd.DataFrame({
    "overlap_at_5": neighbor_overlap_at_k(plain_neighbor_idx, json_neighbor_idx, k=5),
    "overlap_at_10": neighbor_overlap_at_k(plain_neighbor_idx, json_neighbor_idx, k=10),
})

display(neighbor_overlap.describe().T)

plt.figure(figsize=(8.5, 5.2), dpi=180)
ax = sns.histplot(neighbor_overlap["overlap_at_10"], bins=20, kde=True)
ax.set_title("Neighbor overlap between plain text and JSON embeddings", pad=12, weight="bold")
ax.set_xlabel("Top-10 neighbor overlap")
ax.set_ylabel("Number of games")
plt.tight_layout()
plt.show()

### 11.5. Qualitative Comparison for the Same Query Game

Bảng dưới đây so sánh hai chiến lược trên cùng một query game. Với ví dụ `Goat Simulator`, plain text embedding trả về nhiều game có chủ đề simulator/sandbox/funny rõ ràng như `Vacation Simulator` và `Totally Accurate Battle Simulator`.

Structured JSON embedding tạo danh sách khác hơn, có một số game vẫn liên quan đến simulation/sandbox, nhưng cũng xuất hiện kết quả rộng hơn như RPG/MMO. Điều này khớp với metric: JSON embedding tạo neighborhood khác, nhưng chưa giữ tag overlap tốt bằng plain text.


In [ ]:
def compare_recommendations_for_game(local_idx=None, top_n=6):
    if local_idx is None:
        # Pick a game with enough tag information for a meaningful sanity check.
        candidate = embedding_df.sort_values(["TagRichness", "TotalReviews"], ascending=False).head(30)
        local_idx = int(candidate.sample(1, random_state=7).index[0])

    source = embedding_df.iloc[local_idx]
    source_tags = source.get("tag_list", [])

    plain_neighbors = plain_neighbor_idx[local_idx][:top_n]
    json_neighbors = json_neighbor_idx[local_idx][:top_n]

    rows = []
    for representation, neighbor_list in [
        ("Plain text", plain_neighbors),
        ("Structured JSON", json_neighbors),
    ]:
        for rank, neighbor_idx in enumerate(neighbor_list, start=1):
            nbr = embedding_df.iloc[neighbor_idx]
            rows.append({
                "representation": representation,
                "rank": rank,
                "source_game": source["Name"],
                "recommended_game": nbr["Name"],
                "tag_jaccard": round(tag_jaccard(source_tags, nbr.get("tag_list", [])), 3),
                "same_developer": safe_value(source.get("Developers")) == safe_value(nbr.get("Developers")),
                "same_publisher": safe_value(source.get("Publishers")) == safe_value(nbr.get("Publishers")),
                "price_gap": round(abs(float(source.get("price", 0)) - float(nbr.get("price", 0))), 2),
                "bayesian_gap": round(abs(float(source.get("BayesianRating", 0)) - float(nbr.get("BayesianRating", 0))), 4),
                "recommended_tags": ", ".join(nbr.get("tag_list", [])[:8]),
            })

    print("Source game:", source["Name"])
    print("Source tags:", ", ".join(source_tags[:12]))
    return pd.DataFrame(rows)

comparison_example = compare_recommendations_for_game(top_n=6)
display(comparison_example)

### 11.6. Interpretation

Kết quả thực nghiệm cho thấy structured JSON embedding **chưa vượt plain text embedding** trong phiên bản hiện tại.

Cụ thể:

- Plain text embedding tốt hơn về `Tag Jaccard`, same developer và same publisher.
- JSON embedding tạo neighbor khác đáng kể, thể hiện qua overlap@5 và overlap@10 thấp.
- JSON chỉ cải thiện rất nhẹ ở Bayesian rating gap, nhưng chưa đủ để xem là lựa chọn tốt hơn cho recommender nội dung.

Kết luận: serialize game thành JSON là hướng có tiềm năng cho production/vector database, nhưng không nên dùng nguyên JSON compact hiện tại rồi kỳ vọng tự động tốt hơn. 


## 12. Data Readiness Conclusion

Từ EDA, feature engineering và embedding evaluation, có thể rút ra các kết luận chính:

1. **Raw data chưa phù hợp để dùng trực tiếp.**  
   Dữ liệu ban đầu có missing values, chuỗi rỗng, định dạng giá/ngày cần chuẩn hóa, tag/genre chưa nhất quán và các cột text có thể gây lỗi khi lưu CSV rồi đọc lại ở notebook khác.

2. **Clean data đã đủ điều kiện cho web/recommendation prototype.**  
   Các cột cốt lõi như `Name`, `Description`, `price`, `ReleaseDate`, `Developers`, `Publishers`, requirement và `tags` đã được chuẩn hóa để phục vụ search, filter, detail page và recommendation.

3. **Review và popularity có long-tail mạnh.**  
   Không nên dùng raw review count làm ranking chính vì dễ tạo popularity bias. `BayesianRating`, `PopularityScore`, `FreshnessScore` và `DiscoveryScore` giúp cân bằng chất lượng, độ tin cậy, độ mới và khả năng khám phá.

4. **Feature engineering đã tạo được các nhóm tín hiệu thực dụng.**  
   Notebook xây dựng content features, tag richness, tag multi-hot, weighted content text, review quality, freshness, affordability và composite discovery score. Đây là bộ feature đủ tốt để triển khai web version đầu.

5. **Embedding baseline có bằng chứng định lượng tốt.**  
   `ContentText` + TF-IDF/SVD cho nearest-neighbor tốt hơn random baseline rõ rệt: cosine, tag Jaccard, same developer và same publisher đều có lift cao. Điều này chứng minh vector representation giữ được tín hiệu domain, không chỉ là biến đổi số học.

6. **Structured JSON embedding là hướng thử nghiệm, chưa phải lựa chọn tốt nhất hiện tại.**  
   JSON embedding tạo neighborhood khác plain text, nhưng output cho thấy plain text vẫn tốt hơn về tag overlap và production metadata. Vì vậy, trong web prototype nên ưu tiên `ContentText` embedding; JSON document có thể dùng sau khi tinh chỉnh schema/field weighting hoặc dùng semantic embedding model mạnh hơn.

**Final recommendation.**  
Dataset sau preprocessing đã sẵn sàng để xây dựng hệ thống web/recommendation. Pipeline đề xuất cho phiên bản đầu là:

`clean data → engineered features → ContentText embedding → nearest-neighbor retrieval → rerank/filter bằng tags, price, OS, BayesianRating và DiscoveryScore`.

Cách này vừa minh bạch, dễ giải thích, vừa đủ mạnh để tạo recommendation có ý nghĩa trước khi đầu tư vào vector database hoặc semantic embedding phức tạp hơn.
